# Pregunta 5 — Quins directors tenen millor rating i quins recapten més?
**Projecte:** Anàlisi d'un conjunt de dades (0378 – Seguretat i alta disponibilitat, Tema 1)
**Dataset:** `IMDB-Movie-Data.csv` — 1.000 pel·lícules (2006–2016)

Aquest notebook treballa **una única pregunta de recerca**, amb el procediment complet: preparació de les dades, mètode d'anàlisi, resultat i interpretació.

## 1. Preparació de les dades

**Procediment:**
- Carreguem el CSV en un `DataFrame` de Pandas.
- Renombrem columnes amb espais/parèntesis (`runtime`, `revenue`).
- Convertim `Revenue = 0` a `NaN` (no és una recaptació real, és un valor desconegut).
- Separem `genre` i `actors` en llistes (cada pel·lícula en pot tenir diversos).
- Creem un subconjunt `df_fiable` amb `votes >= 1000`, perquè les pel·lícules amb pocs vots tenen una nota poc representativa (sobretot les de 2016).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("IMDB-Movie-Data.csv", encoding="utf-8")
df = df.rename(columns={"Runtime (Minutes)": "runtime", "Revenue (Millions)": "revenue"})
df.columns = [c.lower() for c in df.columns]

df.loc[df["revenue"] == 0, "revenue"] = np.nan

df["genres"] = df["genre"].str.split(",").apply(lambda l: [g.strip() for g in l])
df["actors_list"] = df["actors"].str.split(",").apply(lambda l: [a.strip() for a in l])

df_fiable = df[df["votes"] >= 1000].copy()

print(f"Total de pel·lícules: {len(df)}")
print(f"Amb >= 1000 vots (notes fiables): {len(df_fiable)}")
df.head(3)

## 2. Pregunta de recerca

**Quins directors tenen millor rating mitjà i quins recapten més en total?**

**Mètode:** fem servir `df_fiable` (≥1000 vots). Agrupem per `director` i calculem el nombre de pel·lícules, el rating mitjà i la recaptació total de cadascun. Exigim un mínim de 3 pel·lícules per director perquè la mitjana sigui representativa (un director amb una sola pel·lícula no permet parlar de "mitjana").

In [ ]:
d = (df_fiable.groupby("director")
              .agg(n=("title", "count"),
                   rating_mitja=("rating", "mean"),
                   recaptacio_total=("revenue", "sum"))
              .query("n >= 3")
              .round(2))
print(f"Directors amb almenys 3 pel·lícules: {len(d)}")

### Millor rating mitjà

In [ ]:
d.sort_values("rating_mitja", ascending=False).head(8)

### Més recaptació total

In [ ]:
d.sort_values("recaptacio_total", ascending=False).head(8)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
top_rating = d.sort_values("rating_mitja", ascending=False).head(8)
top_rating["rating_mitja"][::-1].plot(kind="barh", ax=ax[0], color="darkorange")
ax[0].set_title("Directors amb millor rating mitjà (≥3 films)"); ax[0].set_xlim(6, 9)

top_rev = d.sort_values("recaptacio_total", ascending=False).head(8)
top_rev["recaptacio_total"][::-1].plot(kind="barh", ax=ax[1], color="steelblue")
ax[1].set_title("Directors amb més recaptació total (M)")
plt.tight_layout(); plt.show()

## 3. Resultats

- Per rating mitjà destaquen **Christopher Nolan** (8,68, 5 pel·lícules), **Martin Scorsese** (7,92) i **Quentin Tarantino** (7,90).
- Per recaptació total destaquen **J.J. Abrams** (1.683 M), **David Yates** (1.631 M) i **Christopher Nolan** (1.515 M, 3r lloc).
- **Christopher Nolan** és l'únic director que apareix a totes dues llistes de manera destacada: millor nota mitjana i, alhora, entre els que més recapten. La resta de directors amb millor nota (Scorsese, Tarantino) no apareixen entre els que més recapten en total, i viceversa amb Michael Bay o Zack Snyder.


## 4. Interpretació i limitacions

✏️ **Omple-ho tu:** Que Nolan destaqui en nota i en recaptació no és casualitat: manté un segell d’autor reconegut treballant amb pressupostos de blockbuster. Cal recordar que la recaptació total afavoreix qui té més pel·lícules al dataset.